# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from sklearn.model_selection import train_test_split

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Freshness amplifies quality

One finding in the FlyRank research paper is that freshness alone does not guarantee stronger content performance. The paper reports that when word count is considered, fresh content in the 3,500+ word group has a higher health score than older content in the same group.

My methodology question:
I would ask how well the health score represents actual content quality in this comparison. The health score is a FlyRank composite made from impressions, position, CTR, and scroll depth, rather than an external ground-truth quality label. Therefore, the result supports an observed relationship between freshness, content depth, and this composite metric, but it does not prove that refreshing content directly causes better performance. I would also want to check whether content age, topic, or client differences could explain part of the observed difference.

Finding 2: Average position is the strongest predictor of health score

The paper's Random Forest feature-importance analysis found that average position had the highest importance when predicting health score, followed by impressions and scroll depth. However, the paper also clearly notes that health score is partly constructed from these same inputs.

My methodology question:
My main question would be whether this is really a predictive discovery or partly a form of target leakage. Since average position and impressions already contribute to the health score, it is expected that a model would find them important. The validation can show that the model predicts the composite score, but it does not support a causal claim that improving average position will independently cause health score improvement. I would therefore describe this result as descriptive model behavior rather than an optimization rule.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

### Honest validation approach

My Week-5 model evaluates performance using a client-level holdout. Pages from the same client are kept together, so the model is tested on clients that were not used during training.

To show why validation design matters, I first evaluate the same Random Forest model using a random row split. This is the "before" evaluation.

I then evaluate the model using the grouped-by-client split from my Week-5 model. This is the "after" evaluation.

The grouped split is more appropriate for this use case because pages from the same client may share similar content, performance patterns, or other characteristics. A random row split can place pages from the same client in both training and testing, which may make the evaluation look more optimistic than performance on completely unseen clients.

Both evaluations use the same target, feature set, preprocessing approach, Random Forest model, and evaluation metrics.

In [11]:
!git clone https://github.com/sureshbudha879-pixel/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 160 (delta 66), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (160/160), 1.98 MiB | 5.24 MiB/s, done.
Resolving deltas: 100% (66/66), done.


In [12]:
!ls /content/flyrank-ml-internship/data/raw

content_refresh_anonymized.csv


In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Dataset shape:", df.shape)
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

Dataset shape: (30000, 45)
Declining rate: 0.542


In [14]:
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "days_since_last_update"
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level",
    "impression_tier",
    "position_tier"
]

features = numeric_features + categorical_features

print("Number of features:", len(features))
print("Excluded leakage columns:", ["trend_direction", "trend_pct"])

Number of features: 20
Excluded leakage columns: ['trend_direction', 'trend_pct']


In [15]:
def build_pipeline():

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    return pipeline


def precision_at_k(y_true, scores, k):

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(scores)[::-1]

    top_k = order[:k]

    return y_true[top_k].mean()

In [16]:
before_train, before_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["is_declining_label"]
)

X_before_train = before_train[features]
y_before_train = before_train["is_declining_label"]

X_before_test = before_test[features]
y_before_test = before_test["is_declining_label"]

before_pipeline = build_pipeline()

before_pipeline.fit(
    X_before_train,
    y_before_train
)

before_probabilities = before_pipeline.predict_proba(
    X_before_test
)[:, 1]

before_p20 = precision_at_k(
    y_before_test,
    before_probabilities,
    20
)

before_p50 = precision_at_k(
    y_before_test,
    before_probabilities,
    50
)

before_p100 = precision_at_k(
    y_before_test,
    before_probabilities,
    100
)

before_auc = roc_auc_score(
    y_before_test,
    before_probabilities
)

print("BEFORE — Random row split")
print("Precision@20:", round(before_p20, 3))
print("Precision@50:", round(before_p50, 3))
print("Precision@100:", round(before_p100, 3))
print("ROC AUC:", round(before_auc, 3))

BEFORE — Random row split
Precision@20: 1.0
Precision@50: 0.98
Precision@100: 0.97
ROC AUC: 0.743


In [17]:
clients = df["client_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

after_train = df[
    df["client_id"].isin(train_clients)
].copy()

after_test = df[
    df["client_id"].isin(test_clients)
].copy()

print("AFTER — Grouped by client")
print("Training rows:", len(after_train))
print("Test rows:", len(after_test))
print("Training clients:", after_train["client_id"].nunique())
print("Test clients:", after_test["client_id"].nunique())

AFTER — Grouped by client
Training rows: 26581
Test rows: 3419
Training clients: 25
Test clients: 7


In [18]:
X_after_train = after_train[features]
y_after_train = after_train["is_declining_label"]

X_after_test = after_test[features]
y_after_test = after_test["is_declining_label"]

after_pipeline = build_pipeline()

after_pipeline.fit(
    X_after_train,
    y_after_train
)

after_probabilities = after_pipeline.predict_proba(
    X_after_test
)[:, 1]

after_p20 = precision_at_k(
    y_after_test,
    after_probabilities,
    20
)

after_p50 = precision_at_k(
    y_after_test,
    after_probabilities,
    50
)

after_p100 = precision_at_k(
    y_after_test,
    after_probabilities,
    100
)

after_auc = roc_auc_score(
    y_after_test,
    after_probabilities
)

print("Precision@20:", round(after_p20, 3))
print("Precision@50:", round(after_p50, 3))
print("Precision@100:", round(after_p100, 3))
print("ROC AUC:", round(after_auc, 3))

Precision@20: 0.75
Precision@50: 0.72
Precision@100: 0.68
ROC AUC: 0.637


In [20]:
comparison = pd.DataFrame([
    {
        "Validation Method": "Random row split — Before",
        "Precision@20": before_p20,
        "Precision@50": before_p50,
        "Precision@100": before_p100,
        "ROC AUC": before_auc
    },
    {
        "Validation Method": "Grouped by client — After",
        "Precision@20": after_p20,
        "Precision@50": after_p50,
        "Precision@100": after_p100,
        "ROC AUC": after_auc
    }
])

display(comparison.round(3))

,Validation Method,Precision@20,Precision@50,Precision@100,ROC AUC
0,Random row split — Before,1.00,0.98,0.97,0.743
1,Grouped by client — After,0.75,0.72,0.68,0.637


### Interpretation

The random row split and grouped-by-client split use the same model and features, but they answer different validation questions.

The random row split allows pages from the same client to appear in both training and testing. This can make the test set more similar to the training data.

The grouped-by-client split keeps each client's pages together and tests the model on clients not used during training. For this reason, I treat the grouped result as the more relevant measure of generalization for this project.

The difference between the before and after metrics shows how validation design can affect the measured performance of the same model.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I reviewed the final feature set for information that would reveal the target directly or use information that would not be available at decision time.

The main leakage risk is `trend_direction` because my target is created directly from this column:

`is_declining_label = (trend_direction == "down")`

Including `trend_direction` as a model feature would allow the model to directly access the information used to create the answer.

I also excluded `trend_pct` because it is directly connected to the observed trend and could provide information too closely related to the target definition.

I also reviewed identifier columns. `content_id` and `client_id` are not used as model features. `client_id` is used only for the grouped validation split.

The final model feature set contains content, search-performance, engagement, and freshness signals. These are used as input signals rather than direct copies of the target.

In [22]:
leakage_audit = pd.DataFrame([
    {
        "Column": "trend_direction",
        "Used as feature": "No",
        "Reason": "Used directly to create the target label"
    },
    {
        "Column": "trend_pct",
        "Used as feature": "No",
        "Reason": "Directly related to the observed trend and potential target leakage"
    },
    {
        "Column": "content_id",
        "Used as feature": "No",
        "Reason": "Identifier, not a meaningful predictive signal"
    },
    {
        "Column": "client_id",
        "Used as feature": "No",
        "Reason": "Used only for grouped validation"
    }
])

display(leakage_audit)

,Column,Used as feature,Reason
0,trend_direction,No,Used directly to create the target label
1,trend_pct,No,Directly related to the observed trend and pot...
2,content_id,No,"Identifier, not a meaningful predictive signal"
3,client_id,No,Used only for grouped validation


In [23]:
error_examples = after_test[
    [
        "content_id",
        "is_declining_label",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "sessions_90d",
        "days_since_last_update"
    ]
].copy()

error_examples["model_score"] = after_probabilities

error_examples["prediction"] = (
    error_examples["model_score"] >= 0.5
).astype(int)

false_positives = error_examples[
    (error_examples["prediction"] == 1) &
    (error_examples["is_declining_label"] == 0)
].sort_values(
    "model_score",
    ascending=False
)

false_negatives = error_examples[
    (error_examples["prediction"] == 0) &
    (error_examples["is_declining_label"] == 1)
].sort_values(
    "model_score",
    ascending=True
)

print("False positives:")
display(false_positives.head(5))

print("False negatives:")
display(false_negatives.head(5))

False positives:


,content_id,is_declining_label,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,days_since_last_update,model_score,prediction
8990,content_a38c8f61e246,0,447,0,0.00,8.0,4,20,0.933333,1
2882,content_5d64fc00babd,0,470,0,0.00,7.3,2,20,0.913333,1
29401,content_f6a8bb38f970,0,1596,5,0.31,6.2,11,104,0.910000,1
22524,content_846bb4dd8b44,0,870,1,0.11,17.6,2,104,0.906667,1
15705,content_4dd569ee33c9,0,361,0,0.00,14.8,4,20,0.903333,1


False negatives:


,content_id,is_declining_label,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,days_since_last_update,model_score,prediction
27395,content_e72e6c56f0a3,1,1,0,0.0,9.0,1,20,0.026667,0
18171,content_24796d98b025,1,1,0,0.0,2.0,1,20,0.040000,0
25350,content_a4c38287770e,1,2,0,0.0,5.0,1,20,0.046667,0
12364,content_3a4e24a3a6a8,1,1,0,0.0,4.0,1,20,0.063333,0
19104,content_a860ee9e7ae4,1,1,0,0.0,8.0,2,104,0.070000,0


### Error examples

The model does not correctly classify every page.

False positives are pages that received a high predicted probability of decline but did not have the observed declining label.

False negatives are pages that had the observed declining label but received a lower predicted probability from the model.

These examples are useful because they show that the model should not be treated as an automatic decision system. The model provides a ranked signal that can support review and prioritization.

In my Week-5 results, some highly ranked pages had a label of 0 even though the model assigned them high scores. This shows that strong model confidence does not guarantee that every prediction is correct.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original stronger claim

"The Random Forest is better at identifying pages that need to be refreshed."

### Rewritten claim

"On the held-out clients in this evaluation, the Random Forest measured higher ranking performance than the leakage-safe baseline. This is an observed result from the current dataset and validation design. The model should be used as directional decision-support for prioritizing pages for review, not as an automatic decision about whether a page must be refreshed."

This version is more careful because it describes what was observed and measured instead of claiming that the model will always identify the correct pages in every situation.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.